# Module 02: LLMs, SLMs & Multimodal Models

**Companion notebook for**: `02-llms-slms-multimodal.html`

## Learning Objectives

- Understand the difference between Large Language Models (LLMs) and Small Language Models (SLMs)
- Compare closed-weight vs open-weight model families and their tradeoffs
- Explore multimodal capabilities: sending images to vision-enabled models
- Count tokens and estimate API costs across different providers
- Visualize benchmark scores to compare model performance
- Understand quantization and its impact on model size and quality
- Apply a practical decision framework for model selection

## Prerequisites

- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+
- Basic familiarity with the OpenAI Python SDK

---
## Setup & Dependencies

In [ ]:
%pip install -q openai tiktoken matplotlib numpy Pillow

In [ ]:
import os
import base64
import json
import textwrap
from pathlib import Path

import tiktoken
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from openai import OpenAI

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

client = OpenAI()  # uses OPENAI_API_KEY from env
print("All imports successful. OpenAI API key found.")

---
## Section 1: The LLM Landscape — Model Architecture Comparison

The model landscape is defined by three axes: **parameter count**, **training data**, and **compute budget**.
A model's parameter count determines its memory footprint (roughly 2 bytes per parameter in float16),
inference speed, and — with diminishing returns — its reasoning ceiling.

Common size classes in the open-weight world:

| Size | VRAM (fp16) | Use Case | Examples |
|------|------------|----------|----------|
| 3-4B | ~7 GB | Edge / mobile, fast classification | Phi-3-mini, Gemma 2 2B |
| 7-8B | ~14 GB | Consumer GPU, general tasks | Llama 3 8B, Mistral 7B |
| 13-14B | ~28 GB | Strong reasoning on single GPU | Phi-4, Qwen 2.5 14B |
| 70B | ~140 GB | A100-class, near-frontier quality | Llama 3 70B, Qwen 2.5 72B |
| 405B+ | Multi-GPU | Research-grade, frontier quality | Llama 3.1 405B |

Let's build a comparison table and explore how these numbers translate into practice.

In [ ]:
# ----------------------------------------------------------------
# Model Parameter Counting & Architecture Comparison Table
# ----------------------------------------------------------------
# This table summarises the key properties of major model families.
# VRAM estimates assume float16 precision (2 bytes per parameter).

models = [
    # (Name, Params (B), Type, Architecture, Context Window, VRAM fp16 GB, License)
    ("Phi-3-mini",        3.8,  "Open",   "Dense",  128_000, 7.6,   "MIT"),
    ("Gemma 2 2B",        2.0,  "Open",   "Dense",   8_192,  4.0,   "Apache 2.0"),
    ("Gemma 2 9B",        9.0,  "Open",   "Dense",   8_192, 18.0,   "Apache 2.0"),
    ("Llama 3 8B",        8.0,  "Open",   "Dense",  128_000, 16.0,  "Apache 2.0"),
    ("Mistral 7B",        7.0,  "Open",   "Dense",   32_768, 14.0,  "Apache 2.0"),
    ("Phi-4",            14.0,  "Open",   "Dense",   16_384, 28.0,  "MIT"),
    ("Mixtral 8x7B",     46.0,  "Open",   "MoE (2/8)", 32_768, 24.0, "Apache 2.0"),
    ("Llama 3 70B",      70.0,  "Open",   "Dense",  128_000, 140.0, "Apache 2.0"),
    ("Llama 3.1 405B",  405.0,  "Open",   "Dense",  128_000, 810.0, "Apache 2.0"),
    ("GPT-4o-mini",       None, "Closed", "Unknown", 128_000, None,  "Proprietary"),
    ("GPT-4o",            None, "Closed", "Unknown", 128_000, None,  "Proprietary"),
    ("Claude Sonnet",     None, "Closed", "Unknown", 200_000, None,  "Proprietary"),
    ("Gemini 2.0 Flash",  None, "Closed", "Unknown",1_000_000,None,  "Proprietary"),
]

# Pretty-print the table
header = f"{'Model':<22} {'Params':>8} {'Type':<8} {'Arch':<12} {'Context':>10} {'VRAM fp16':>10} {'License':<14}"
print(header)
print("-" * len(header))
for name, params, mtype, arch, ctx, vram, lic in models:
    p = f"{params:.1f}B" if params else "N/A"
    v = f"{vram:.0f} GB" if vram else "N/A"
    c = f"{ctx:,}"
    print(f"{name:<22} {p:>8} {mtype:<8} {arch:<12} {c:>10} {v:>10} {lic:<14}")

In [ ]:
# ----------------------------------------------------------------
# Mixture-of-Experts (MoE) — Active vs Total Parameters
# ----------------------------------------------------------------
# MoE models have many "expert" FFN blocks but only activate a
# subset per token. This gives quality of a large model with
# the inference speed of a much smaller one.

moe_example = {
    "model": "Mixtral 8x7B",
    "total_params_B": 46.0,
    "num_experts": 8,
    "experts_active_per_token": 2,
    "active_params_B": 12.0,  # approximate
}

print("Mixture-of-Experts Example: Mixtral 8x7B")
print(f"  Total parameters:           {moe_example['total_params_B']:.0f}B")
print(f"  Number of experts:           {moe_example['num_experts']}")
print(f"  Experts active per token:    {moe_example['experts_active_per_token']}")
print(f"  Active parameters per token: ~{moe_example['active_params_B']:.0f}B")
print()
print("Implication: Mixtral 8x7B has quality closer to a ~40B dense model")
print("             but inference speed closer to a ~12B dense model.")

---
## Section 2: LLM vs SLM — Comparing Responses

A **Large Language Model** (like GPT-4o) has more parameters and broader knowledge,
while a **Small Language Model** (like GPT-4o-mini) trades some capability for
dramatically lower cost and faster inference.

Let's send the same prompt to both and compare quality, latency, and cost.

In [ ]:
import time

# ----------------------------------------------------------------
# Compare LLM (gpt-4o) vs SLM (gpt-4o-mini) on the same prompt
# ----------------------------------------------------------------

PROMPT = (
    "Explain the difference between supervised learning and "
    "reinforcement learning. Give one real-world example of each. "
    "Keep your answer under 150 words."
)

MODELS_TO_COMPARE = [
    {"name": "gpt-4o-mini", "label": "SLM (GPT-4o-mini)",
     "price_in": 0.15, "price_out": 0.60},  # per million tokens
    {"name": "gpt-4o",      "label": "LLM (GPT-4o)",
     "price_in": 2.50, "price_out": 10.00},
]


def call_and_measure(model_name: str, prompt: str) -> dict:
    """Call a model and return response text, token counts, and wall-clock time."""
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    elapsed = time.perf_counter() - start
    usage = response.usage
    return {
        "text": response.choices[0].message.content,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "latency_s": round(elapsed, 2),
    }


print(f"Prompt: {PROMPT}\n")
print("=" * 70)

for spec in MODELS_TO_COMPARE:
    result = call_and_measure(spec["name"], PROMPT)

    # Estimate cost
    cost_in  = (result["input_tokens"]  / 1_000_000) * spec["price_in"]
    cost_out = (result["output_tokens"] / 1_000_000) * spec["price_out"]
    total    = cost_in + cost_out

    print(f"\n>>> {spec['label']}  (latency: {result['latency_s']}s)")
    print(f"    Tokens — in: {result['input_tokens']}, out: {result['output_tokens']}")
    print(f"    Est. cost: ${total:.6f}")
    print()
    # Wrap text for readability
    for line in textwrap.wrap(result["text"], width=78):
        print(f"    {line}")
    print("=" * 70)

**Key take-away:** For a bounded, well-defined task like this, the SLM (GPT-4o-mini)
produces a perfectly adequate answer at roughly **17x lower cost** and noticeably
lower latency. The LLM (GPT-4o) may produce a slightly more nuanced answer, but
the difference is often negligible for production use cases that do not require
frontier reasoning.

SLMs shine for: classification, extraction, summarisation, simple Q&A, and
high-volume pipelines. LLMs are necessary for: complex reasoning, multi-step
code generation, nuanced instruction following, and research-level tasks.

---
## Section 3: Token Counting & Cost Estimation

Before sending a request to an API, you can count tokens locally using
`tiktoken` and estimate the cost. This is essential for budgeting, especially
when processing large documents or running high-volume pipelines.

In [ ]:
# ----------------------------------------------------------------
# Token counting with tiktoken
# ----------------------------------------------------------------

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Count tokens for OpenAI-family models using tiktoken."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))


def estimate_cost(
    prompt: str,
    model: str = "gpt-4o",
    estimated_output_tokens: int = 300,
) -> dict:
    """Estimate the cost of a single API call before sending it."""
    # Prices per 1M tokens (input / output) — early 2025
    PRICING = {
        "gpt-4o":             (2.50,  10.00),
        "gpt-4o-mini":        (0.15,   0.60),
        "claude-sonnet-4-6":  (3.00,  15.00),
        "gemini-2.0-flash":   (0.075,  0.30),
    }

    input_tokens = count_tokens(prompt, model)
    price_in, price_out = PRICING.get(model, (3.0, 15.0))

    input_cost  = (input_tokens / 1_000_000) * price_in
    output_cost = (estimated_output_tokens / 1_000_000) * price_out

    return {
        "model": model,
        "input_tokens": input_tokens,
        "estimated_output_tokens": estimated_output_tokens,
        "input_cost_usd": round(input_cost, 8),
        "output_cost_usd": round(output_cost, 8),
        "total_est_usd": round(input_cost + output_cost, 8),
    }


# --- Demo: estimate cost for a long prompt across multiple models ---
long_prompt = (
    "Summarize the following research paper on transformer architectures, "
    "focusing on the key innovations and experimental results. "
) + "Lorem ipsum dolor sit amet. " * 200  # ~1000 words of filler

print(f"Prompt length: {len(long_prompt):,} characters\n")

for model_name in ["gpt-4o", "gpt-4o-mini", "claude-sonnet-4-6", "gemini-2.0-flash"]:
    est = estimate_cost(long_prompt, model=model_name)
    print(f"  {model_name:<22}  tokens: {est['input_tokens']:>6,}  "
          f"est. cost: ${est['total_est_usd']:.6f}")

In [ ]:
# ----------------------------------------------------------------
# Scale analysis: cost at volume
# ----------------------------------------------------------------
# What does it cost to run 1 million requests per day?

daily_requests = 1_000_000
avg_input_tokens = 500
avg_output_tokens = 300

pricing = {
    "GPT-4o":           (2.50, 10.00),
    "GPT-4o-mini":      (0.15,  0.60),
    "Claude Sonnet":    (3.00, 15.00),
    "Gemini 2.0 Flash": (0.075, 0.30),
    "Self-hosted 7B":   (0.00,  0.00),   # hardware cost only
}

print(f"Cost comparison: {daily_requests:,} requests/day")
print(f"  Average {avg_input_tokens} input tokens + {avg_output_tokens} output tokens per request\n")

header = f"{'Model':<22} {'Daily Cost':>12} {'Monthly Cost':>14} {'Annual Cost':>14}"
print(header)
print("-" * len(header))

for name, (p_in, p_out) in pricing.items():
    total_in  = daily_requests * avg_input_tokens
    total_out = daily_requests * avg_output_tokens
    daily_cost = (total_in / 1e6) * p_in + (total_out / 1e6) * p_out
    monthly = daily_cost * 30
    annual  = daily_cost * 365
    if daily_cost == 0:
        print(f"{name:<22} {'~$60*':>12} {'~$1,800*':>14} {'~$21,900*':>14}")
    else:
        print(f"{name:<22} ${daily_cost:>10,.0f} ${monthly:>12,.0f} ${annual:>12,.0f}")

print("\n* Self-hosted estimate assumes a single A100 cloud instance at ~$2.50/hr.")
print("  Actual cost depends on hardware, quantisation, and utilisation.")

---
## Section 4: Multimodal Demo — Sending an Image to GPT-4o Vision

Multimodal models can process **text + images** in a single request. The image
is split into 16x16 pixel patches, encoded by a Vision Transformer (ViT),
projected into the LLM's embedding space, and concatenated with the text tokens
to form a unified input sequence.

Practical use cases include:
- Screenshot debugging (paste an error screenshot and ask for a fix)
- Document extraction (send a PDF page image, get structured JSON)
- Chart/diagram analysis
- Handwriting recognition

Below we demonstrate two approaches:
1. Sending a **URL-based image** (no file needed)
2. Sending a **base64-encoded local image**

In [ ]:
# ----------------------------------------------------------------
# Multimodal: send a URL-based image to GPT-4o
# ----------------------------------------------------------------
# We use a public sample image (a simple chart) to demonstrate.

IMAGE_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/47/PNG_transparency_demonstration_1.png/"
    "280px-PNG_transparency_demonstration_1.png"
)

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Describe this image in detail. "
                        "What objects do you see and what colours are present?"
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": IMAGE_URL},
                },
            ],
        }
    ],
    max_tokens=500,
)

print("GPT-4o Vision Response:\n")
for line in textwrap.wrap(response.choices[0].message.content, width=78):
    print(f"  {line}")

print(f"\nTokens used — prompt: {response.usage.prompt_tokens}, "
      f"completion: {response.usage.completion_tokens}")

In [ ]:
# ----------------------------------------------------------------
# Multimodal: send a local base64-encoded image to GPT-4o
# ----------------------------------------------------------------
# This pattern works for any local file — screenshots, PDFs
# rendered as images, photos, etc.

def send_image_to_gpt4o(image_path: str, prompt: str, model: str = "gpt-4o") -> str:
    """
    Send a local image file to GPT-4o vision and return the response text.

    Args:
        image_path: Path to a local image file (PNG, JPEG, GIF, WEBP).
        prompt:     Text prompt to accompany the image.
        model:      Model name (must support vision).

    Returns:
        The model's text response.
    """
    path = Path(image_path)
    suffix = path.suffix.lower()
    media_types = {".png": "image/png", ".jpg": "image/jpeg",
                   ".jpeg": "image/jpeg", ".gif": "image/gif",
                   ".webp": "image/webp"}
    media_type = media_types.get(suffix, "image/png")

    image_b64 = base64.standard_b64encode(path.read_bytes()).decode("utf-8")
    data_url = f"data:{media_type};base64,{image_b64}"

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }
        ],
        max_tokens=1024,
    )
    return response.choices[0].message.content


# Example usage (uncomment and set a real path to test locally):
# result = send_image_to_gpt4o("chart.png", "Extract all data from this chart as JSON.")
# print(result)

print("send_image_to_gpt4o() defined.")
print("Usage: send_image_to_gpt4o('path/to/image.png', 'Your prompt here')")

### Image Token Cost

Images are not free — they consume tokens in the pricing model:
- A 512x512 image uses approximately **170 tokens** (OpenAI).
- A 1024x1024 image uses approximately **680 tokens**.
- Maximum image size: 20 MB for GPT-4o, 5 MB for Claude.

For high-volume document processing, resizing images to the minimum
resolution that preserves readability can significantly reduce costs.

---
## Section 5: Benchmark Comparison Visualisation

Benchmarks provide a rough ranking of model capabilities. The most commonly
cited benchmarks include:

- **MMLU** — 57-subject knowledge test (human expert ~89%)
- **HumanEval** — Python code generation
- **GPQA** — Graduate-level STEM questions (very hard, contamination-resistant)
- **MATH** — Competition-level mathematics

Important caveat: benchmark scores are useful for rough comparisons but do not
reliably predict performance on your specific production task. Always build a
golden eval set for your use case.

In [ ]:
# ----------------------------------------------------------------
# Benchmark Comparison Chart (matplotlib)
# ----------------------------------------------------------------
# Scores are approximate and sourced from public benchmarks as of
# early 2025. They illustrate relative positioning, not exact values.

benchmark_data = {
    "Model": [
        "GPT-4o",
        "Claude Sonnet 3.7",
        "Gemini 1.5 Pro",
        "Llama 3 70B",
        "Mixtral 8x7B",
        "Phi-4 (14B)",
        "Llama 3 8B",
        "Phi-3-mini (3.8B)",
    ],
    "MMLU": [88.7, 88.0, 85.9, 82.0, 70.6, 78.2, 66.6, 69.0],
    "HumanEval": [90.2, 89.0, 84.1, 81.7, 63.4, 82.6, 62.2, 58.5],
    "GPQA": [53.6, 51.0, 46.2, 39.5, 29.0, 42.1, 26.8, 24.0],
    "MATH": [76.6, 75.0, 67.7, 50.4, 36.8, 58.3, 30.0, 32.5],
}

models_list = benchmark_data["Model"]
benchmarks = ["MMLU", "HumanEval", "GPQA", "MATH"]
n_models = len(models_list)
n_benchmarks = len(benchmarks)

x = np.arange(n_models)
bar_width = 0.18
colours = ["#6366f1", "#22d3ee", "#f59e0b", "#22c55e"]

fig, ax = plt.subplots(figsize=(14, 6))

for i, bench in enumerate(benchmarks):
    offset = (i - n_benchmarks / 2 + 0.5) * bar_width
    bars = ax.bar(
        x + offset,
        benchmark_data[bench],
        bar_width,
        label=bench,
        color=colours[i],
        alpha=0.85,
        edgecolor="white",
        linewidth=0.5,
    )

ax.set_xlabel("Model", fontsize=12, fontweight="bold")
ax.set_ylabel("Score (%)", fontsize=12, fontweight="bold")
ax.set_title("Model Benchmark Comparison (early 2025)", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=30, ha="right", fontsize=10)
ax.set_ylim(0, 100)
ax.legend(loc="upper right", fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: GPQA scores are notably lower across all models — this benchmark")
print("      tests graduate-level STEM reasoning where even PhD experts score ~65%.")

In [ ]:
# ----------------------------------------------------------------
# Capability vs Cost scatter plot (value frontier)
# ----------------------------------------------------------------

scatter_data = [
    # (name, avg_benchmark_score, cost_per_1M_output_tokens)
    ("GPT-4o",              77.3, 10.00),
    ("Claude Sonnet 3.7",   75.8, 15.00),
    ("Gemini 1.5 Pro",      71.0,  5.00),
    ("Llama 3 70B",         63.4,  0.80),   # typical API pricing
    ("Mixtral 8x7B",        50.0,  0.50),
    ("GPT-4o-mini",         62.0,  0.60),
    ("Gemini 2.0 Flash",    60.0,  0.30),
    ("Phi-4 (14B)",         65.3,  0.00),   # self-hosted
    ("Llama 3 8B",          46.4,  0.00),   # self-hosted
    ("Phi-3-mini (3.8B)",   46.0,  0.00),   # self-hosted
]

fig, ax = plt.subplots(figsize=(10, 6))

for name, score, cost in scatter_data:
    color = "#22c55e" if cost == 0 else ("#6366f1" if cost > 5 else "#22d3ee")
    ax.scatter(cost, score, s=120, color=color, edgecolors="white", linewidth=0.8, zorder=5)
    ax.annotate(name, (cost, score), textcoords="offset points",
                xytext=(8, 5), fontsize=8.5, color="#333")

ax.set_xlabel("Cost per 1M Output Tokens ($)", fontsize=11, fontweight="bold")
ax.set_ylabel("Average Benchmark Score (%)", fontsize=11, fontweight="bold")
ax.set_title("Capability vs Cost — The Value Frontier", fontsize=13, fontweight="bold")
ax.grid(alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#6366f1", markersize=10, label="Premium API"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#22d3ee", markersize=10, label="Budget API"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#22c55e", markersize=10, label="Self-hosted (free)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

plt.tight_layout()
plt.show()

---
## Section 6: Quantization Concepts

**Quantization** reduces the numerical precision of model weights to shrink memory
usage and speed up inference — often with minimal quality loss.

| Precision | Bytes/Param | 7B Model Size | Quality Loss (MMLU) |
|-----------|-------------|---------------|---------------------|
| float32   | 4           | 28 GB         | baseline            |
| float16   | 2           | 14 GB         | ~0%                 |
| int8      | 1           | 7 GB          | ~1-3%               |
| int4      | 0.5         | 3.5 GB        | ~3-7%               |

The dominant format for quantized local models is **GGUF**, used by the
`llama.cpp` inference engine and Ollama. A single command like
`ollama run llama3.2` will download a pre-quantized model and start serving it.

Ollama also exposes an OpenAI-compatible REST API on `localhost:11434`,
so you can swap between local and cloud models by changing only the base URL.

In [ ]:
# ----------------------------------------------------------------
# Quantization: memory savings calculator
# ----------------------------------------------------------------

def quantization_summary(params_billion: float) -> None:
    """Print a memory comparison table for different quantization levels."""
    precisions = [
        ("float32", 4.0,  "Baseline — rarely used for inference"),
        ("float16", 2.0,  "Standard inference precision"),
        ("int8",    1.0,  "Good balance of size and quality"),
        ("int4",    0.5,  "Aggressive — runs on consumer GPUs"),
    ]

    print(f"Quantization impact for a {params_billion:.1f}B parameter model:\n")
    header = f"{'Precision':<12} {'Bytes/Param':>12} {'Model Size':>12} {'vs fp32':>10}"
    print(header)
    print("-" * len(header))

    base_size = params_billion * 4  # float32 baseline in GB
    for name, bpp, note in precisions:
        size_gb = params_billion * bpp
        ratio = size_gb / base_size
        print(f"{name:<12} {bpp:>12.1f} {size_gb:>10.1f} GB {ratio:>9.0%}")
    print()


# Show for popular model sizes
for size in [3.8, 7.0, 14.0, 70.0]:
    quantization_summary(size)
    print()

In [ ]:
# ----------------------------------------------------------------
# Ollama local call — OpenAI-compatible SDK pattern
# ----------------------------------------------------------------
# This code works with Ollama running locally. It uses the same
# OpenAI SDK — you just change the base_url and model name.
# Uncomment to test if you have Ollama installed.

def call_local_model(prompt: str, model: str = "llama3.2") -> str:
    """
    Call a local model via Ollama's OpenAI-compatible API.

    Prerequisites:
        - Install Ollama: https://ollama.com
        - Pull a model: ollama pull llama3.2
        - Ollama server running on localhost:11434
    """
    local_client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama",  # required by SDK but ignored by Ollama
    )

    response = local_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    return response.choices[0].message.content


# Uncomment to test (requires Ollama running locally):
# result = call_local_model("Explain quantization in two sentences.")
# print(result)

print("call_local_model() defined.")
print("To test: install Ollama, run 'ollama pull llama3.2', then uncomment the lines above.")

---
## Section 7: Model Selection Decision Framework

There is **no single best model**. The right model depends on your specific
constraints: latency, cost, quality requirements, data sensitivity, and
whether you need multimodal capabilities.

### The Latency-Cost-Quality Triangle

You can optimise for **two of three** properties, but not all three:

- **Fast + Cheap** (trade quality) — GPT-4o-mini, Gemini Flash, Claude Haiku
- **Fast + High Quality** (trade cost) — GPT-4o, Claude Sonnet
- **High Quality + Cheap** (trade latency) — Self-hosted Llama 70B

### Decision Tree

```
START
  |
  +-- Need < 500ms latency?
  |     YES --> GPT-4o-mini / Gemini Flash / Claude Haiku / local SLM
  |     NO
  |      |
  |      +-- Need vision / multimodal?
  |      |     YES --> GPT-4o / Claude 3+ / Gemini 1.5+ / Llama 3.2
  |      |     NO
  |      |      |
  |      |      +-- Complex reasoning / math / code?
  |      |      |     YES --> Claude Sonnet / GPT-4o / o1-o3
  |      |      |     NO
  |      |      |      |
  |      |      |      +-- Private data / on-premises required?
  |      |      |      |     YES --> Llama 3 / Mistral / Phi-4 via Ollama
  |      |      |      |     NO  --> GPT-4o-mini / Gemini Flash (best value)
```

In [ ]:
# ----------------------------------------------------------------
# Model Selection Decision Framework — programmatic version
# ----------------------------------------------------------------

def recommend_model(
    needs_low_latency: bool = False,
    needs_vision: bool = False,
    needs_complex_reasoning: bool = False,
    needs_on_premises: bool = False,
    budget_sensitive: bool = False,
) -> dict:
    """
    Recommend a model based on task requirements.

    Returns a dict with primary recommendation, alternatives,
    and the reasoning behind the choice.
    """
    if needs_on_premises:
        if needs_vision:
            return {
                "primary": "Llama 3.2 11B Vision",
                "alternatives": ["LLaVA 1.6", "Qwen-VL"],
                "reason": "On-premises + vision narrows choices to open-weight multimodal models.",
                "hosting": "Ollama or vLLM on local GPU (24+ GB VRAM)",
            }
        if needs_complex_reasoning:
            return {
                "primary": "Llama 3 70B",
                "alternatives": ["Qwen 2.5 72B", "DeepSeek-R1 distilled 14B"],
                "reason": "Complex reasoning on-premises requires a large open-weight model.",
                "hosting": "vLLM on A100-class hardware (140+ GB VRAM)",
            }
        return {
            "primary": "Phi-4 (14B)",
            "alternatives": ["Llama 3 8B", "Mistral 7B", "Gemma 2 9B"],
            "reason": "General on-premises tasks are best served by efficient SLMs.",
            "hosting": "Ollama on consumer GPU (16+ GB VRAM)",
        }

    if needs_low_latency:
        return {
            "primary": "GPT-4o-mini" if not budget_sensitive else "Gemini 2.0 Flash",
            "alternatives": ["Claude Haiku", "Gemini 2.0 Flash", "GPT-4o-mini"],
            "reason": "Sub-500ms latency requires fast, lightweight API models.",
            "hosting": "Cloud API",
        }

    if needs_vision:
        return {
            "primary": "GPT-4o",
            "alternatives": ["Claude Sonnet (strong at documents)", "Gemini 1.5 Pro (native multimodal)"],
            "reason": "Vision tasks need multimodal-capable frontier models.",
            "hosting": "Cloud API",
        }

    if needs_complex_reasoning:
        return {
            "primary": "Claude Sonnet",
            "alternatives": ["GPT-4o", "o1/o3 (for frontier difficulty)"],
            "reason": "Complex reasoning benefits from frontier model capabilities.",
            "hosting": "Cloud API",
        }

    # Default: general purpose, cost-effective
    return {
        "primary": "Gemini 2.0 Flash" if budget_sensitive else "GPT-4o-mini",
        "alternatives": ["GPT-4o-mini", "Gemini 2.0 Flash"],
        "reason": "General tasks are best served by cost-effective models with good quality.",
        "hosting": "Cloud API",
    }


# --- Demo: run through several scenarios ---
scenarios = [
    {"desc": "Real-time autocomplete for a chat UI",
     "args": {"needs_low_latency": True, "budget_sensitive": True}},
    {"desc": "Extracting tables from scanned PDF invoices",
     "args": {"needs_vision": True}},
    {"desc": "Generating complex SQL from natural language",
     "args": {"needs_complex_reasoning": True}},
    {"desc": "Classifying patient records (HIPAA-regulated)",
     "args": {"needs_on_premises": True}},
    {"desc": "High-volume email categorisation (10M/day)",
     "args": {"budget_sensitive": True}},
]

for scenario in scenarios:
    rec = recommend_model(**scenario["args"])
    print(f"Scenario: {scenario['desc']}")
    print(f"  Recommendation:  {rec['primary']}")
    print(f"  Alternatives:    {', '.join(rec['alternatives'])}")
    print(f"  Reason:          {rec['reason']}")
    print(f"  Hosting:         {rec['hosting']}")
    print()

### Professional Testing Methodology

The correct way to choose a model for production is **not** to look at benchmarks.
It is to build a **golden evaluation set** specific to your task and test
candidates systematically:

1. **Define** your task precisely and collect 50-200 representative examples.
2. **Label** these examples with correct outputs (human-annotated or from a trusted model).
3. **Run** each candidate model on all examples with your production prompt.
4. **Score** outputs against golden labels (exact match, ROUGE, BERTScore, or LLM-as-judge).
5. **Sort** models by quality-per-dollar on your specific task.
6. **Pick** the model at the knee of the curve — where additional quality costs disproportionately more.

This methodology produces decisions that are **defensible**, **reproducible**,
and often surprising in which model wins.

---
## Summary

In this notebook we covered:

1. **Model Architecture Comparison** — Parameter counts, VRAM requirements, and the MoE architecture
2. **LLM vs SLM Responses** — Side-by-side comparison of GPT-4o vs GPT-4o-mini showing quality, latency, and cost
3. **Token Counting & Cost Estimation** — Using tiktoken to budget API costs before sending requests
4. **Multimodal Vision** — Sending images to GPT-4o via URL and base64 encoding
5. **Benchmark Visualisation** — MMLU, HumanEval, GPQA, and MATH scores across 8 models
6. **Quantization** — How int8/int4 quantization shrinks models to fit on consumer hardware
7. **Model Selection Framework** — A decision tree and programmatic tool for choosing the right model

**Key Principle**: There is no single best model. Model selection is a practical engineering
decision based on latency, cost, quality, data sensitivity, and modality requirements.
Start with the best model to set a quality ceiling, then test cheaper alternatives to
find the optimal tradeoff for your specific task.

**Next notebook**: `03-api-for-llms.ipynb` — Working with LLM APIs, authentication, streaming, and function calling.